In [8]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

In [9]:
df = pd.read_csv('C:/Users/andre/OneDrive/Ambiente de Trabalho/dashoard_UPT/DataSet_GroupProject/DATASET/dataset_limpo_medianag.csv')
# Reemplazar "Sem dado" por NaN
df = df.replace("Sem dado", np.nan)

# Quitar filas vacías en columnas clave
df = df.dropna(subset=['DevType', 'YearsCodePro'])

In [10]:
dev_counts = df['DevType'].value_counts().head(10)

fig_career_paths = go.Figure([
    go.Bar(
        x=dev_counts.values,
        y=dev_counts.index,
        orientation='h',
        text=dev_counts.values,
        textposition='outside'
    )
])

fig_career_paths.update_layout(
    template='plotly_white',
    height=500
)

In [11]:
# Top 5 + Others
top5 = df['DevType'].value_counts().nlargest(5)
others = df['DevType'].value_counts().iloc[5:].sum()

career_clean = top5.copy()
career_clean['Others'] = others

fig_clean = go.Figure([
    go.Pie(
        labels=career_clean.index,
        values=career_clean.values,
        hole=0.4
    )
])

fig_clean.update_layout(
    template='plotly_white',
    height=400
)

In [12]:
# Convertir a número
df['YearsCodePro'] = pd.to_numeric(df['YearsCodePro'], errors='coerce')

# Crear rangos
bins = [0, 2, 5, 10, 20, 50]
labels = ['0-2', '3-5', '6-10', '11-20', '20+']

df['ExperienceLevel'] = pd.cut(df['YearsCodePro'], bins=bins, labels=labels)

exp_counts = df['ExperienceLevel'].value_counts().sort_index()

fig_exp = go.Figure([
    go.Bar(
        x=exp_counts.index,
        y=exp_counts.values,
        text=exp_counts.values,
        textposition='outside'
    )
])

fig_exp.update_layout(
    template='plotly_white',
    height=400
)

In [13]:
top3 = df['DevType'].value_counts().head(3).index
df_top = df[df['DevType'].isin(top3)]

fig_combo = px.box(
    df_top,
    x='DevType',
    y='YearsCodePro',
)

fig_combo.update_layout(
    template='plotly_white',
    height=400
)

In [14]:
# Limpiar datos clave
df_clean = df.dropna(subset=['EdLevel', 'Employment'])

# Top categorías (para no saturar)
top_edu = df_clean['EdLevel'].value_counts().head(4).index
top_emp = df_clean['Employment'].value_counts().head(4).index

df_sankey = df_clean[
    df_clean['EdLevel'].isin(top_edu) &
    df_clean['Employment'].isin(top_emp)
]

# Crear combinaciones
flow = df_sankey.groupby(['EdLevel', 'Employment']).size().reset_index(name='count')

# Crear nodos
labels = list(pd.concat([flow['EdLevel'], flow['Employment']]).unique())

source = flow['EdLevel'].apply(lambda x: labels.index(x))
target = flow['Employment'].apply(lambda x: labels.index(x))

fig_sankey = go.Figure(go.Sankey(
    node=dict(label=labels),
    link=dict(
        source=source,
        target=target,
        value=flow['count']
    )
))

fig_sankey.update_layout(
    template='plotly_white',
    height=500
)

In [16]:
# Limpiar
df_sat = df.dropna(subset=['DevType', 'JobSat'])

# Top 5 DevTypes
top_dev = df_sat['DevType'].value_counts().head(5).index
df_sat = df_sat[df_sat['DevType'].isin(top_dev)]

# Promedio de satisfacción
sat_avg = df_sat.groupby('DevType')['JobSat'].mean().sort_values()

fig_sat = go.Figure([
    go.Bar(
        x=sat_avg.values,
        y=sat_avg.index,
        orientation='h',
        text=round(sat_avg.values, 2),
        textposition='outside'
    )
])

fig_sat.update_layout(
    template='plotly_white',
    height=400
)

In [19]:
fig_sankey.write_html('career_flow.html')
fig_sat.write_html('career_satisfaction.html')